# 5 · AI Agents for Finance

**Outcome of this session:** a working investment-analyst agent, your finished tool published on GitHub, and your capstone submission.

**The storyline, completed.** Session 2: you valued Apple against its peers. Session 3: you learned to prove such a number right. Session 4: your screening engine rebuilt the universe live from filings, as a workflow whose every step you wrote. Today the last capability arrives: a model that **plans its own steps** — it decides which company to fetch first, when to compare, when it has enough. You will grant that autonomy the way a firm grants a mandate: with explicit rules, hard limits, and a human signature at the end. Then you publish the whole tool.

**In this notebook you will:**

- Read a complete agent loop and locate every governance lever in it
- Build the tool and the rules that protect an agent from a currency trap
- Run a governed agent on Apple against Sony — then on a company pair of your choice
- Run the same discipline through PydanticAI, an industry agent framework, and map it to LangGraph
- Publish your finance tool on GitHub


## Workflow vs agent: the distinction that matters

| | Workflow (Session 4) | Agent (now) |
|---|---|---|
| Plan | fixed, written by you | **chosen by the model**, step by step |
| Tools | called by your code | **requested** by the model, executed by your code |
| Stops when | the script ends | the model judges the goal met, or it reaches the limits you set |
| Failure mode | a step errors loudly | wanders, loops, or is confidently wrong |

**An agent is a loop.** Model proposes a tool call → your code executes it → the result goes back → repeat. Governance is not a policy document; it's these levers *in the code*: `max_steps`, a token budget, a tool whitelist, forced structured output, a human gate, and an audit trail.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: anatomy of the agent
We import the course agent and inspect its specification (rules, tools, limits) before granting it any autonomy. Review the specification, not only the output.

In [ ]:
sys.path.insert(0, str(ROOT / "session-05-agents" / "demo"))
import mini_analyst_agent as agent
from toolkit import llm            # for llm.show(): renders long output readably

print(agent.SYSTEM)
print("TOOLS:", [t["name"] for t in agent.TOOLS])
print(f"Guardrails: max_steps={agent.MAX_STEPS}, token_budget={agent.TOKEN_BUDGET:,}")

Read `run_agent` in `session-05-agents/demo/mini_analyst_agent.py` once, carefully. It is roughly forty lines, and it contains the entire concept. Note where each guardrail lives, and that tool errors are sent back to the model (it can adapt), while a malformed recommendation is rejected and returned for correction (validation at the boundary: notebook 04's lesson, applied to the agent's own output).

## Part B: LAB: build the agent's tool and rules

### Exercise 1: the comparison tool with the currency trap

The session's task continues the Apple storyline with its oldest consumer-electronics rival: **Sony**. Sony is listed in New York but files with the SEC in **Japanese yen**. A naive agent compares Sony's ¥13 trillion of revenue to Apple's $416 billion and declares Sony "30 times bigger". Your tool must detect differing units, add an explicit `warning` key for the model, and `print()` it so the human watching the trace sees the trap being caught.

In [ ]:
def tool_compare_metrics(args: dict) -> str:
    """Growth and margins side-by-side, computed in code. args = {"tickers": [...]}."""
    rows = [agent._metrics_row(t) for t in args["tickers"]]
### START CODE HERE ###
    units = {r[None] for r in rows}                       # which field differs across foreign filers?
    out = {"companies": rows}
    if len(units) > None:                                 # warn when how many distinct units?
        out["warning"] = (f"UNITS DIFFER ({', '.join(sorted(units))}): absolute amounts are NOT "
                          "comparable. Compare growth and margins only, or convert currency first.")
        print(f"⚠️ compare_metrics: UNITS DIFFER ({', '.join(sorted(units))}) - flagged to the agent")
### END CODE HERE ###
    return json.dumps(out, indent=1)

_ = tool_compare_metrics({"tickers": ["SONY", "AAPL"]})

In [ ]:
# ✅ self-check: run me (live EDGAR, no API key needed)
out = json.loads(tool_compare_metrics({"tickers": ["SONY", "AAPL"]}))
assert "warning" in out, "SONY reports in JPY, AAPL in USD - your tool must add the warning key"
assert "JPY" in out["warning"] and "USD" in out["warning"]
same = json.loads(tool_compare_metrics({"tickers": ["AAPL", "MSFT"]}))
assert "warning" not in same, "same-currency pairs should NOT warn"
print("All checks passed ✅ - the trap is armed")

### Exercise 2: the agent's operating rules

Write the system prompt. It must cover:

- plan first
- gather every company before comparing
- **always check the `unit` field; never compare absolute amounts across currencies**
- use `compare_metrics` for arithmetic
- every figure comes from tool results only
- `insufficient_data` is an acceptable stance
- finish with `record_recommendation`, exactly once
- this is coursework, not investment advice

In [ ]:
### START CODE HERE ###
MY_SYSTEM = """You are an investment research agent for an educational exercise.

Operating rules:
1. PLAN first: say briefly which tools you will call and why.
2. Gather financials for EVERY company involved before comparing anything.
3. ALWAYS check the '[WHICH FIELD?]' field. Never compare absolute amounts across
   different [ACROSS WHAT?] - compare unitless measures (growth, margins) and say so.
4. Use [WHICH TOOL?] for arithmetic. Do not compute numbers yourself.
5. Every figure you state must come from a tool result in this conversation.
6. If data is missing or ambiguous, use stance '[WHICH STANCE?]'.
7. Finish by calling record_recommendation EXACTLY ONCE. This is coursework,
   not investment advice.
"""
### END CODE HERE ###
print(MY_SYSTEM)

In [ ]:
# ✅ self-check: run me
low = MY_SYSTEM.lower()
for needle, hint in [("unit", "rule about checking the unit field"),
                     ("curren", "forbid cross-currency absolute comparisons"),
                     ("compare_metrics", "arithmetic goes through the tool"),
                     ("record_recommendation", "must finish with the structured recommendation"),
                     ("insufficient", "allow an insufficient_data stance"),
                     ("not investment advice", "coursework disclaimer")]:
    assert needle in low, f"missing: {hint}"
print("All checks passed ✅")

### Exercise 3: run your agent

Three cells, one per idea: **wire** your rules and your tool into the course loop; **state the request** — the single user prompt the agent receives; then **run** and follow the trace: the plan, the tool calls, the moment the currency check fires, and the structured recommendation.

In [ ]:
### START CODE HERE ###
agent.SYSTEM = None                               # wire in YOUR rules from Exercise 2
agent.TOOL_IMPLS["compare_metrics"] = None        # wire in YOUR tool from Exercise 1
### END CODE HERE ###
print("Wired: the loop now runs YOUR rules and YOUR tool.")

**The request.** Everything the agent does starts from one user prompt — this is the whole instruction it receives beyond your system rules. Read it, then swap in your own pair later.

In [ ]:
TASK = ("Analyze Apple (ticker AAPL) and compare it with Sony (ticker SONY): "
        "which is better positioned on growth and profitability?")
print("USER PROMPT >", TASK)
# Your own run later: replace with ANY pair of SEC filers (pharma, banking or tech).

In [ ]:
if HAS_KEY:
    rec, trace = agent.run_agent(TASK)
    if rec:
        llm.show(
            f"**{rec['headline']}**\n\n"
            f"*Stance: {rec['stance']} · Confidence: {rec['confidence']}*\n\n"
            "**Key points**\n\n"
            + "\n".join(f"- {p}" for p in rec["key_points"])
            + "\n\n**Risks**\n\n"
            + "\n".join(f"- {r}" for r in rec["risks"])
            + f"\n\n**What would change this view:** {rec['what_would_change_my_mind']}",
            title="The agent's recommendation")
        print(f"{len(trace)} tool call(s). Token usage: {llm.usage_summary()}")
else:
    print("No API key - run `python session-05-agents/demo/mini_analyst_agent.py --preflight` "
          "in the terminal to study the agent's spec instead.")

In [ ]:
# The HUMAN GATE: nothing is saved until you approve. Read the recommendation
# above. If - and only if - you'd sign it, set APPROVE = True and run.
APPROVE = False

if HAS_KEY and APPROVE and rec:
    OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
    (OUTD / "agent_memo.md").write_text(agent.render_memo(TASK, rec, trace))
    print("Saved outputs/agent_memo.md - with the full audit trail of every tool call.")
elif HAS_KEY:
    print("Not saved. The human gate is a feature, not a formality.")

### Stretch (optional): give the agent a second real tool

Your agent has one analytical tool. Real research agents carry several — and every added tool widens what the agent can decide to do, which is why the tool whitelist is a governance lever. The cell below adds Session 4's revenue forecaster as a second tool, extends the token budget for the extra run (limits are explicit, so widening one is a visible, deliberate act), and repeats the task. Watch two things in the trace: whether the agent *chooses* to call `forecast_revenue`, and whether it respects the holdout error the tool reports alongside the forecast.

**How does the model know which tool to use?** Only through what you register. Every request carries the full tool list — each tool's `name`, `description`, and input schema — and the model reads the descriptions the way you read documentation, then answers with a `tool_use` block naming the tool and its arguments; your code executes and returns the result. The description *is* the interface: ours says the forecast is "reported WITH its measured holdout error", which is exactly what nudges the model to weigh that error. A vague description produces wrong tool choices — writing them is prompt engineering, aimed at a machine that reads carefully.

In [ ]:
# A second real tool: Session 4's least-squares forecaster, packaged for the agent.
if HAS_KEY:
    import numpy as np
    from toolkit import edgar

    def tool_forecast_revenue(args: dict) -> str:
        """FY+1 revenue forecast with its measured holdout error. args = {"ticker": "..."}."""
        recs = edgar.annual_financials(args["ticker"], n=6)["revenue"]
        rev = np.array([v["val"] for v in recs], dtype=float)
        years = np.arange(len(rev), dtype=float)
        slope, intercept = np.polyfit(years[:-1], rev[:-1], 1)
        holdout = abs(slope * years[-1] + intercept - rev[-1]) / rev[-1]
        slope, intercept = np.polyfit(years, rev, 1)
        fc = slope * (years[-1] + 1) + intercept
        return json.dumps({"ticker": args["ticker"],
                           "fy1_revenue_forecast_bn": round(fc / 1e9, 1),
                           "holdout_error": round(float(holdout), 3),
                           "note": "trust the forecast only if holdout_error is small"})

    agent.TOOLS.append({"name": "forecast_revenue",
                        "description": "FY+1 revenue forecast for one ticker (least-squares on six "
                                       "years of filings), reported WITH its measured holdout error.",
                        "input_schema": {"type": "object",
                                         "properties": {"ticker": {"type": "string"}},
                                         "required": ["ticker"]}})
    agent.TOOL_IMPLS["forecast_revenue"] = tool_forecast_revenue
    agent.TOKEN_BUDGET += 100_000   # a second run needs budget - widening a limit is an explicit act

    TASK2 = TASK + " Include next year's revenue outlook where a forecast is available."
    print("USER PROMPT >", TASK2, "\n")
    rec2, trace2 = agent.run_agent(TASK2)
    called = [t for t in trace2 if "forecast_revenue" in t]
    print(f"\nforecast_revenue called: {'yes' if called else 'no - the agent judged it unnecessary; rerun or sharpen the task'}")
    if rec2:
        llm.show(f"**{rec2['headline']}**", title="With the second tool")
    print(f"Token usage: {llm.usage_summary()}")
else:
    print("No API key - read the cell: the pattern is TOOLS.append + TOOL_IMPLS[...] = your function.")

**The two forecasts, drawn.** Each panel stays in the company's own reporting currency — the y-axes are deliberately not comparable (that is the currency lesson in one picture); the growth *rates* are.

In [ ]:
# Six actual fiscal years and the FY+1 forecast, per company, in its own currency.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from toolkit import edgar

panels = {}
for t in ["AAPL", "SONY"]:
    fin = edgar.annual_financials(t, n=6)
    recs, unit = fin["revenue"], fin["unit"]
    rev = np.array([v["val"] for v in recs], dtype=float)
    yy = np.arange(len(rev), dtype=float)
    slope, intercept = np.polyfit(yy[:-1], rev[:-1], 1)
    holdout = abs(slope * yy[-1] + intercept - rev[-1]) / rev[-1]
    slope, intercept = np.polyfit(yy, rev, 1)
    fc = slope * (yy[-1] + 1) + intercept
    panels[t] = {"years": [int(v["fy_end"][:4]) for v in recs], "rev": rev,
                 "fc": fc, "holdout": holdout, "unit": unit,
                 "fy_end": recs[-1]["fy_end"]}

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"{t} ({p['unit']}) — holdout error {p['holdout']:.0%}" for t, p in panels.items()])
for i, (t, p) in enumerate(panels.items()):
    fig.add_trace(go.Scatter(x=p["years"], y=p["rev"] / 1e9, mode="lines+markers",
                             showlegend=False), row=1, col=i + 1)
    fig.add_trace(go.Scatter(x=[p["years"][-1], p["years"][-1] + 1],
                             y=[p["rev"][-1] / 1e9, p["fc"] / 1e9],
                             mode="lines+markers", line=dict(dash="dot"),
                             marker=dict(symbol="diamond"), showlegend=False), row=1, col=i + 1)
fig.update_layout(height=380,
                  title_text="Revenue in billions of each company's OWN currency: actual (solid), FY+1 forecast (dotted)")
fig.show()

## The forward look: what a revenue forecast implies

A revenue forecast becomes financially meaningful one step further on: hold each company's latest net margin constant, and the FY+1 revenue forecast implies FY+1 earnings. This is scenario arithmetic, not a valuation — margins move, and the holdout error already told you how much weight the fitted line can bear — but it is exactly how an analyst begins translating a growth view into an earnings view. Every quantity stays in the company's own currency, so no unit is ever crossed.

In [ ]:
for t, p in panels.items():
    fin = edgar.annual_financials(t, n=6)
    ni_map = {v["fy_end"]: v["val"] for v in fin["net_income"]}
    ni = ni_map.get(p["fy_end"])
    if ni is None:
        print(f"{t}: no net income for {p['fy_end']} - skipped")
        continue
    margin = ni / p["rev"][-1]
    implied = p["fc"] * margin
    print(f"{t:5} ({p['unit']})  FY+1 revenue {p['fc']/1e9:10,.1f}bn x net margin {margin:6.1%} "
          f"= implied FY+1 net income {implied/1e9:9,.1f}bn  ({implied/ni-1:+.1%} vs last year)")
print("\nScenario arithmetic, not a valuation: margin held constant, straight-line revenue.")

## From tools to skills (optional, like the stretch)

A tool gives the agent an *action* it can take; a **skill** gives it *knowledge on demand* — a procedure, a house style, a checklist — packaged as a file the model loads only when the task calls for it.

**Why not simply put the knowledge in the system prompt?** Three reasons, and they are the whole case for skills:

1. **Cost.** The system prompt is sent — and paid for — on every request, needed or not. A skill's one-line description costs a few tokens; its body costs tokens only on the runs that actually use it.
2. **Scale.** A desk owns many procedures: memo style, escalation rules, sector conventions. As a library of skills the agent carries all of them and loads the one the task needs; as system-prompt text they would crowd each other out.
3. **Ownership.** A skill is a file. The desk edits it, git versions it, and every agent that can read it picks up the change — no agent code and no prompt is touched.

**What it adds here, concretely:** the desk's memo house style — five rules — lives in `session-05-agents/skills/memo-style/SKILL.md`. Your system prompt says nothing about style; the only trace of the skill the agent ever sees up front is the `read_skill` tool's one-line description. Watch the run: the agent loads the style *before* writing, and the recommendation comes out obeying rules that exist only in that file — compare the headline with your earlier run. The mechanism is **progressive disclosure**: description always visible, body loaded when chosen — the same economics as a tool call.

This is exactly how the industry versions work. Claude Code discovers `SKILL.md` folders and loads them when relevant; the Claude API runs Agent Skills inside its code-execution container (`container={"skills": [...]}`). Same idea in both: the description is the trigger, the body is the payload.

In [ ]:
# The skill mechanism, raw: one tool that serves knowledge on demand.
SKILLS_DIR = ROOT / "session-05-agents" / "skills"

def tool_read_skill(args: dict) -> str:
    """Return the full text of a skill by name."""
    path = SKILLS_DIR / args["name"] / "SKILL.md"
    return path.read_text() if path.exists() else f"unknown skill: {args['name']}"

print(tool_read_skill({"name": "memo-style"}))   # what the agent will load - five rules

if HAS_KEY:
    agent.TOOLS.append({
        "name": "read_skill",
        "description": "Load the full text of an available skill before the task that needs it. "
                       "Available: 'memo-style' - the desk's house style for investment memos; "
                       "read it BEFORE writing any recommendation.",
        "input_schema": {"type": "object", "properties": {"name": {"type": "string"}},
                         "required": ["name"]},
    })
    agent.TOOL_IMPLS["read_skill"] = tool_read_skill
    agent.TOKEN_BUDGET += 100_000   # another run, another explicit budget extension

    TASK3 = TASK + " Write the recommendation in the desk's house style."
    print("\nUSER PROMPT >", TASK3, "\n")
    rec3, trace3 = agent.run_agent(TASK3)
    print("\nskill loaded:", "yes" if any("read_skill" in t for t in trace3)
          else "no - rerun; the nudge lives entirely in the tool description")
    if rec3:
        llm.show(f"**{rec3['headline']}**\n\n**What would change this view:** "
                 f"{rec3['what_would_change_my_mind']}",
                 title="In house style - compare this headline with your earlier run")
else:
    print("\nNo API key - the mechanism is still visible above: description as trigger, body as payload.")

## From your forty lines to the industry frameworks

You built the loop raw so that no vendor terminology would ever be opaque to you. Now meet the names you will hear in interviews, and notice that you already know what they do.

**PydanticAI** packages exactly what you just built: an agent loop with typed, validated output and usage limits, from the team behind the Pydantic library you have used all course. The cell below is your Exercise 3 agent again, in about fifteen lines — **and it uses your own Exercise 1 tool, unchanged**. Map each piece to your own code: `output_type` is your forced `record_recommendation` schema, `instructions` is your system prompt, `tools=[...]` is your tool whitelist, `UsageLimits(request_limit=...)` is your `MAX_STEPS`, and the typed `result.output` is your validated recommendation.

Watch the run: the framework agent faces the same Apple-versus-Sony task, calls your tool, and your currency warning fires inside someone else's loop. That is the point of writing tools well — they outlive the harness around them.

In [ ]:
try:
    from pydantic_ai import Agent
except ModuleNotFoundError:          # first run on an env set up before pydantic-ai joined the course
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic-ai-slim[anthropic]"], check=True)
    from pydantic_ai import Agent

from pydantic import BaseModel
from typing import Literal
from pydantic_ai.usage import UsageLimits

def compare_metrics(tickers: list[str]) -> str:
    """Growth and margins side-by-side, computed in code, for a list of tickers.
    Check the 'unit' field: foreign filers report in local currency."""
    return tool_compare_metrics({"tickers": tickers})   # YOUR Exercise 1 tool, unchanged

class QuickTake(BaseModel):
    stance: Literal["prefer_first", "prefer_second", "balanced", "insufficient_data"]
    rationale: str

if HAS_KEY:
    framework_agent = Agent(
        "anthropic:claude-sonnet-5",
        output_type=QuickTake,                     # your forced schema
        instructions="You are an equity analyst. Use ONLY tool results. Never compare "
                     "absolute amounts across currencies - compare growth and margins "
                     "and say so. Coursework, not investment advice.",   # your rules
        tools=[compare_metrics],                   # your tool whitelist
    )
    # Notebooks already run an event loop, so we await the agent instead of
    # calling run_sync (which would raise "event loop is already running").
    FRAMEWORK_TASK = ("Compare Apple (AAPL) with Sony (SONY): which is better "
                      "positioned on growth and profitability?")
    print("USER PROMPT >", FRAMEWORK_TASK, "\n")
    result = await framework_agent.run(
        FRAMEWORK_TASK,
        usage_limits=UsageLimits(request_limit=4),  # the framework's MAX_STEPS
    )
    print("stance   :", result.output.stance)       # typed, validated, like your rec
    llm.show(result.output.rationale, title="PydanticAI, same discipline, fifteen lines")
else:
    print("No API key - read the cell instead: every argument maps to a lever you built.")

**LangGraph** you have already used: Session 4's screening engine ended as a LangGraph `StateGraph`. An *agent* in LangGraph is the same idea with one structural change — an edge that loops back, so the model can decide to act again. Every concept maps onto something you wrote:

| Your forty lines | PydanticAI | LangGraph |
|---|---|---|
| the `while` loop | the `Agent` run | a graph with a **loop edge** |
| `messages` list (the conversation is the memory) | `message_history` | the state object + **checkpointer** (persistent memory) |
| forced `record_recommendation` schema | `output_type=Model` | a typed state schema |
| `MAX_STEPS`, `TOKEN_BUDGET` | `UsageLimits` | recursion limits |
| your human gate before saving | deferred tool approval | **`interrupt()`** before a node |
| the audit trail you print | `result.all_messages()` | the checkpointed state history |

**Which one do you reach for?** Choose by the shape of the system:

- **PydanticAI** when the deliverable is *one governed agent* — a deciding loop with typed output, tools and usage limits, in a few lines. That is this session's task — one agent, one validated recommendation — which is why the live runs here use PydanticAI.
- **LangGraph** when the deliverable is a *process* — several steps of mixed code and model work that need an explicit order, checkpoints, retries, or a human interrupt between nodes. That is Session 4's screening pipeline, which is why it is a LangGraph graph.

The levers are the same in both; the question is whether the model's loop *is* the system (PydanticAI) or one node *inside* a larger system (LangGraph).

One honest engineering note, because you will meet it at work: **LangChain**, the ecosystem LangGraph belongs to, connects to Claude through a package that currently requires an older version of the official SDK than PydanticAI does — the two frameworks cannot share one Python environment today. Version conflicts like this are ordinary AI engineering. Because you know the concepts framework-free, a dependency pin is an inconvenience to you, not a wall.

Two professional conclusions. First, every row above still has to be *decided* by you, whichever column you build in: the framework implements the levers, but choosing them remains your work. Second, when a vendor demo says "guardrails, memory, human-in-the-loop", you can now ask the precise question: *which limit, stored where, interrupting what?*

## When not to use an agent

For a repeatable task (reconciliation, screens, reports), a **workflow** is the better choice: cheaper, testable, auditable. For open-ended research whose path depends on the data, an agent fits, with the set of controls you just built. If a regulator asks *"why did it do that?"*, you want either the workflow's fixed plan or the agent's audit trail as your answer. Choose the degree of autonomy per task; it is a setting you control, not a property of the technology.

## Publish your tool

Everything you built these two days is one coherent tool: grounded prompts, a valuation engine, sanity checks, an evidence verifier, a live screening workflow, a governed agent. It becomes a portfolio piece the moment it is public under your name. In Session 2 you created the repository; finish it now.

**1. A README is the difference between code and a tool.** Create `README.md` at the root of your repository:

```markdown
# AI Equity Research Toolkit

Built during IESE's AI-Augmented Productivity for Finance course.

## What it does
- Values a company against its peers from live SEC EDGAR data
- Screens a 16-company universe on growth and margins - criteria as dials
- Writes grounded investment rationales and audits every number in them
- Runs a governed research agent with hard limits and a human gate

## Trust features
- Every model output is validated at the boundary (Pydantic schemas)
- Every figure in generated prose is traced to source data or flagged
- Nothing is saved without explicit human approval

## Run it
pip install -r requirements.txt   # then add your keys to .env
jupyter lab notebooks/
```

Adjust it to what YOUR repository actually contains — a README that overpromises fails the course's one rule.

**2. Commit and push:**

```bash
git add -A
git commit -m "Screening workflow, governed agent, generated memos"
git push
```

(If you skipped the repository step in Session 2: `gh repo create my-finance-toolkit --private --source . --push`, or github.com → New repo → follow "push an existing repository".)

**3. Check it renders.** Open the repository page in your browser. That URL goes on the capstone submission — and on your CV.

## Deliverable checklist

- [ ] Both ✅ checks green; your agent ran end-to-end on a pair YOU chose
- [ ] The JPY/USD warning appeared in a trace at least once
- [ ] You can point to every guardrail in the code (max_steps, budget, whitelist, forced output, gate, audit trail)
- [ ] `outputs/agent_memo.md` saved through the human gate
- [ ] Your repository is on GitHub with a README — the finished tool, public under your name
- [ ] Capstone submitted: repository URL posted in the class chat (brief + memo skeleton: `session-05-agents/capstone/capstone-brief.md`)

**You now own the whole stack:** grounded prompts → comparable-company valuation → tests & evidence verification → a screening workflow on live SEC data, in LangGraph → a governed agent. *Never ship a number you haven't verified.* A good next step: choose one recurring task at your desk, build it as a workflow, and show a colleague.